## Imports

In [1]:
import os
import re
import time
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)

In [3]:
## Connection
connection = presto.connect(
        
        host='presto-gateway.processing.data.production.internal',
#     presto.processing.yoda.run
#     presto-gateway.processing.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
cwd = os.getcwd()
print(cwd)
local_datasource = '/Users/E2074/local-datasets/customer/loyalty/experiment1'
print(local_datasource)

/Users/E2074/analytics/customer/loyalty/experiment1
/Users/E2074/local-datasets/customer/loyalty/experiment1


## Datasets & Parameter

In [5]:
## Parameter 
current_date = datetime.now()
start_date = '20250110'
end_date = '20250409'

city = 'Bangalore'

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')
segment_date = end_dt.strftime('%Y-%m-%d')
print(start_date, ' to ' ,end_date)
print(segment_date)

20250110  to  20250409
2025-04-09


### Gross Customers

In [6]:
data_query = f"""

WITH customer_base as (
    
    SELECT 
        customer_id,
        city_name,
        customer_obj_mobile customer_mobile,
        
        COUNT(DISTINCT order_id) gross_count,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara

    FROM    
        orders.order_logs_fact
    WHERE  
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND city_name = 'Bangalore'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','Bike Metro','CabEconomy','CabPremium')
    GROUP BY 1,2,3
    ),
    
    segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '{segment_date}'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    )
    
    SELECT 
        customer_base.*,
        coalesce( ltr_segment, 'UNKNOWN')ltr_segment,
        coalesce( retention_segment, 'UNKNOWN') retention_segment,
        coalesce( lifetime_stage, 'UNKNOWN') lifetime_stage,
        coalesce( service_affinity, 'UNKNOWN') service_affinity,
        coalesce( gender, 'UNKNOWN') gender
    FROM 
        customer_base
    LEFT JOIN 
        segment
        ON customer_base.customer_id = segment.customer_id
    WHERE 
        LENGTH(customer_mobile) = 10
"""

# Execute the query and get the result as a DataFrame
# df_data = pd.read_sql(data_query, connection)

# df_data.to_parquet( local_datasource + '/customer_data_{}_{}.parquet'.format(start_date, end_date), index=False)

In [7]:
df_data = pd.read_parquet( local_datasource + '/customer_data_{}_{}.parquet'.format(start_date, end_date))

In [8]:
print(df_data.shape)
df_data.head(5)

(5377708, 13)


,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender
0,5d2e99ac01197b4e4d1a7385,Bangalore,7337328783,4,3,0,0,0,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN
1,5d67a9f63576981183979156,Bangalore,8197947717,10,4,0,2,3,PHH,SILVER,HOOK,UNKNOWN,MALE
2,65d21d619c95be0a095920c5,Bangalore,6363177376,212,2,26,7,174,PHH,DORMANT,DORMANT,ONLY_AUTO,MALE
3,67b2baf8707c9f9a4dc713df,Bangalore,8123445185,10,9,0,0,0,PHH,DORMANT,DORMANT,UNKNOWN,MALE
4,61c9d08a0bfea368f81a0fc7,Bangalore,7483599402,20,18,0,0,2,PHH,DORMANT,DORMANT,ONLY_AUTO,FEMALE


In [9]:
df_data['mobile_length'] = df_data['customer_mobile'].astype(str).str.len()

In [10]:
df_data[df_data['mobile_length'] == 10].shape

(5377708, 14)

## User defined function

In [11]:
agg_dict = {
    'customers': ('customer_id', 'nunique'),
    'gross_count': ('gross_count', 'sum'),
    'net_count': ('net_count', 'sum'),
    'cobrm': ('cobrm', 'sum'),
    'cobra': ('cobra', 'sum'),
    'ocara': ('ocara', 'sum'),
}

def calculate_percentage(df, numerator, denominator):
    
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    
    return percentage

def calculate_percentage_columns(df):
        
    df['g2n'] = calculate_percentage(df, 'net_count', 'gross_count')
    df['cobrm %'] = calculate_percentage(df, 'cobrm', 'gross_count')
    df['cobra %'] = calculate_percentage(df, 'cobra', 'gross_count')
    df['ocara %'] = calculate_percentage(df, 'ocara', 'gross_count')
    
    return df

## Sampling

In [12]:
df_data.head(2)

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,mobile_length
0,5d2e99ac01197b4e4d1a7385,Bangalore,7337328783,4,3,0,0,0,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,10
1,5d67a9f63576981183979156,Bangalore,8197947717,10,4,0,2,3,PHH,SILVER,HOOK,UNKNOWN,MALE,10


In [13]:
df_data.columns

Index(['customer_id', 'city_name', 'customer_mobile', 'gross_count',
       'net_count', 'cobrm', 'cobra', 'ocara', 'ltr_segment',
       'retention_segment', 'lifetime_stage', 'service_affinity', 'gender',
       'mobile_length'],
      dtype='object')

In [14]:
df_analysis2 = df_data\
.groupby(['city_name'])\
.agg(**agg_dict)\
.reset_index()

df_analysis2 = calculate_percentage_columns(df_analysis2)
df_analysis2

,city_name,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,5377453,73674555,43327440,665519,12641609,12152662,58.81,0.9,17.16,16.5


In [15]:
df_data['city_name'].value_counts(True)

city_name
Bangalore    1.0
Name: proportion, dtype: float64

In [16]:
df_data.shape

(5377708, 14)

In [17]:
df_data.customer_id.nunique()

5377453

In [18]:
duplicated_values = df_data['customer_id'].duplicated(keep=False) 
df_cleaned = df_data[~duplicated_values]
df_cleaned.shape

(5377198, 14)

In [19]:
df_cleaned.customer_id.nunique()

5377198

In [20]:
# random_seed = 42

# df_train_control, df_test = train_test_split(df_cleaned, 
#                                              test_size=0.3, 
#                                              random_state=random_seed
#                                             )

# df_train_control = df_train_control.sample(len(df_train_control), random_state=random_seed)
# df_test = df_test.sample(len(df_test), random_state=random_seed)

# # Assign group labels
# df_train_control['group_tc'] = 'control'
# df_test['group_tc'] = 'test'

# df_final_sample = pd.concat([df_train_control, df_test])


random_seed = 42
n_sample = 450000  # 5L

# Shuffle the dataset first
df_cleaned = df_cleaned.sample(frac=1, random_state=random_seed).reset_index(drop=True)

# Take 5L for control
df_control = df_cleaned.iloc[:n_sample].copy()
df_control['group_tc'] = 'control'

# Take next 5L for test
df_test = df_cleaned.iloc[n_sample:n_sample*2].copy()
df_test['group_tc'] = 'test'

# Combine the final sample
df_final_sample = pd.concat([df_control, df_test], ignore_index=True)

In [21]:
df_final_sample['group_tc'].value_counts(True)

group_tc
control    0.5
test       0.5
Name: proportion, dtype: float64

In [22]:
df_final_sample.groupby('city_name')['group_tc'].value_counts(True).unstack(fill_value=0)

group_tc,control,test
city_name,,
Bangalore,0.5,0.5


In [23]:
df_final_sample.columns

Index(['customer_id', 'city_name', 'customer_mobile', 'gross_count',
       'net_count', 'cobrm', 'cobra', 'ocara', 'ltr_segment',
       'retention_segment', 'lifetime_stage', 'service_affinity', 'gender',
       'mobile_length', 'group_tc'],
      dtype='object')

In [24]:
df1 = df_final_sample\
.groupby(['city_name','group_tc'])\
.agg(**agg_dict)\
.reset_index()

df1 = calculate_percentage_columns(df1)
df1

,city_name,group_tc,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,control,450000,6146426,3615750,54703,1056736,1013306,58.83,0.89,17.19,16.49
1,Bangalore,test,450000,6157214,3620063,55268,1053547,1018116,58.79,0.90,17.11,16.54


In [25]:
df2 = df_final_sample\
.groupby(['city_name','ltr_segment', 'group_tc'])\
.agg(**agg_dict)\
.reset_index()

df2 = calculate_percentage_columns(df2)
df2

,city_name,ltr_segment,group_tc,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,HH,control,85958,225500,125156,3171,34485,42417,55.50,1.41,15.29,18.81
1,Bangalore,HH,test,87074,226303,126641,3094,34433,42235,55.96,1.37,15.22,18.66
2,Bangalore,PHH,control,307308,5571631,3306996,44497,959425,900133,59.35,0.80,17.22,16.16
3,Bangalore,PHH,test,306512,5584959,3311782,44615,956414,905645,59.30,0.80,17.12,16.22
4,Bangalore,UNKNOWN,control,56734,349295,183598,7035,62826,70756,52.56,2.01,17.99,20.26
5,Bangalore,UNKNOWN,test,56414,345952,181640,7559,62700,70236,52.50,2.18,18.12,20.30


In [26]:
df3 = df_final_sample\
.groupby(['city_name','retention_segment', 'group_tc'])\
.agg(**agg_dict)\
.reset_index()

df3 = calculate_percentage_columns(df3)
df3

,city_name,retention_segment,group_tc,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,DORMANT,control,161046,922770,553469,9850,146481,155149,59.98,1.07,15.87,16.81
1,Bangalore,DORMANT,test,161222,914476,550034,9374,144463,153177,60.15,1.03,15.80,16.75
2,Bangalore,ELITE,control,32191,2642138,1608120,17573,439837,421131,60.86,0.67,16.65,15.94
3,Bangalore,ELITE,test,32464,2667403,1619151,18358,441989,427740,60.70,0.69,16.57,16.04
4,Bangalore,GOLD,control,54767,625543,363885,5476,111896,98934,58.17,0.88,17.89,15.82
5,Bangalore,GOLD,test,54732,625020,363071,5487,110780,99210,58.09,0.88,17.72,15.87
6,Bangalore,HH,control,29370,90388,52158,1000,14476,14310,57.70,1.11,16.02,15.83
7,Bangalore,HH,test,29677,90225,52403,1108,14465,14030,58.08,1.23,16.03,15.55
8,Bangalore,INACTIVE,control,20272,41168,0,1180,14521,17643,0.00,2.87,35.27,42.86
9,Bangalore,INACTIVE,test,20426,41414,0,1101,14554,17855,0.00,2.66,35.14,43.11


In [27]:
df4 = df_final_sample\
.groupby(['city_name','lifetime_stage', 'group_tc'])\
.agg(**agg_dict)\
.reset_index()

df4 = calculate_percentage_columns(df4)
df4

,city_name,lifetime_stage,group_tc,customers,gross_count,net_count,cobrm,cobra,ocara,g2n,cobrm %,cobra %,ocara %
0,Bangalore,CHURN_OTB,control,14061,303638,188899,2558,49393,46158,62.21,0.84,16.27,15.20
1,Bangalore,CHURN_OTB,test,13959,300506,187689,2334,48500,45357,62.46,0.78,16.14,15.09
2,Bangalore,COMMITTED,control,57157,3243330,1935923,23177,555668,521566,59.69,0.71,17.13,16.08
3,Bangalore,COMMITTED,test,57665,3276971,1950569,23897,558714,531001,59.52,0.73,17.05,16.20
4,Bangalore,DETOX,control,14,524,283,0,133,55,54.01,0.00,25.38,10.50
5,Bangalore,DETOX,test,34,891,504,7,219,95,56.57,0.79,24.58,10.66
6,Bangalore,DORMANT,control,161046,922770,553469,9850,146481,155149,59.98,1.07,15.87,16.81
7,Bangalore,DORMANT,test,161222,914476,550034,9374,144463,153177,60.15,1.03,15.80,16.75
8,Bangalore,HANDHOLDING,control,29370,90388,52158,1000,14476,14310,57.70,1.11,16.02,15.83
9,Bangalore,HANDHOLDING,test,29677,90225,52403,1108,14465,14030,58.08,1.23,16.03,15.55


In [28]:
df_final_sample.groupby(['city_name', 'retention_segment'])['group_tc'].value_counts(True).unstack(fill_value=0)

group_tc                      control      test
city_name retention_segment                    
Bangalore DORMANT            0.499727  0.500273
          ELITE              0.497889  0.502111
          GOLD               0.500160  0.499840
          HH                 0.497400  0.502600
          INACTIVE           0.498108  0.501892
          PLATINUM           0.499643  0.500357
          PRIME              0.509167  0.490833
          SILVER             0.503138  0.496862
          UNKNOWN            0.501414  0.498586

### Export

In [29]:
df_final_sample.head(5)

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,mobile_length,group_tc
0,67a62a5739dd551af0eddb23,Bangalore,6202742044,28,5,1,8,6,PHH,DORMANT,DORMANT,ONLY_CAB,MALE,10,control
1,61d00a0da757467724e4a143,Bangalore,7975903747,14,10,0,3,1,PHH,GOLD,SOFT_CHURN,ONLY_LINK,MALE,10,control
2,63e4ea9fec0ea850e32cdbd1,Bangalore,7845811202,11,3,0,4,3,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,10,control
3,65ae2b4b35ebfff90f5fc774,Bangalore,9019388700,9,4,0,2,2,PHH,SILVER,HOOK,UNKNOWN,MALE,10,control
4,6768ee424e77d8694bd7b467,Bangalore,7047699844,6,4,0,1,1,PHH,GOLD,HOOK,UNKNOWN,MALE,10,control


In [30]:
df_final_sample.shape

(900000, 15)

In [31]:
df_final_sample.to_parquet(local_datasource + '/test_control.parquet', index=False)

In [24]:
df_final_sample = pd.read_parquet(local_datasource + '/test_control.parquet')

In [25]:
df_export_control = df_final_sample[df_final_sample['group_tc'] == 'control']
df_export = df_final_sample[df_final_sample['group_tc'] == 'test']

In [26]:
df_export_control.shape, df_export.shape

((450000, 15), (450000, 15))

In [27]:
display(df_export_control.head(5))
display(df_export.head(5))

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,mobile_length,group_tc
0,67a62a5739dd551af0eddb23,Bangalore,6202742044,28,5,1,8,6,PHH,DORMANT,DORMANT,ONLY_CAB,MALE,10,control
1,61d00a0da757467724e4a143,Bangalore,7975903747,14,10,0,3,1,PHH,GOLD,SOFT_CHURN,ONLY_LINK,MALE,10,control
2,63e4ea9fec0ea850e32cdbd1,Bangalore,7845811202,11,3,0,4,3,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,10,control
3,65ae2b4b35ebfff90f5fc774,Bangalore,9019388700,9,4,0,2,2,PHH,SILVER,HOOK,UNKNOWN,MALE,10,control
4,6768ee424e77d8694bd7b467,Bangalore,7047699844,6,4,0,1,1,PHH,GOLD,HOOK,UNKNOWN,MALE,10,control


,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,mobile_length,group_tc
450000,663b4f0aad80a754bf3a81eb,Bangalore,9025479400,1,1,0,0,0,PHH,DORMANT,DORMANT,UNKNOWN,FEMALE,10,test
450001,6795f673c46f6c4f7eef9e9c,Bangalore,7892031156,5,5,0,0,0,PHH,DORMANT,DORMANT,UNKNOWN,MALE,10,test
450002,65bdf23de65856c258cc311b,Bangalore,7797556557,74,67,0,4,3,PHH,ELITE,COMMITTED,ONLY_AUTO,MALE,10,test
450003,5e494571ab02656b352834d7,Bangalore,9051855749,99,65,0,23,9,PHH,ELITE,COMMITTED,NO_AFFINITY,MALE,10,test
450004,5d5bc0b2bde10e0d54496ffa,Bangalore,9390267572,8,4,0,3,1,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,10,test


In [28]:
df_export[['customer_mobile']].iloc[:500001]

,customer_mobile
450000,9025479400
450001,7892031156
450002,7797556557
450003,9051855749
450004,9390267572
...,...
899995,7904025232
899996,9916362966
899997,9003261320
899998,9731142819


In [29]:
overlap = df_export_control['customer_mobile'].isin(df_export['customer_mobile']).any()

if overlap:
    print("❌ DataFrames are NOT mutually exclusive.")
else:
    print("✅ DataFrames are mutually exclusive.")

❌ DataFrames are NOT mutually exclusive.


In [30]:
# Convert to sets and check for intersection
set1 = set(df_export_control['customer_mobile'])
set2 = set(df_export['customer_mobile'])

if set1.isdisjoint(set2):
    print("✅ DataFrames are mutually exclusive.")
else:
    print("❌ DataFrames are NOT mutually exclusive.")

❌ DataFrames are NOT mutually exclusive.


In [31]:
df_overlap = df_export_control[df_export_control['customer_mobile'].isin(df_export['customer_mobile'])][['customer_mobile']]

In [32]:
df_export_control = df_export_control[~df_export_control['customer_mobile'].isin(df_overlap['customer_mobile'])]

In [33]:
df_export = df_export[~df_export['customer_mobile'].isin(df_overlap['customer_mobile'])]

In [34]:
overlap = df_export_control['customer_mobile'].isin(df_export['customer_mobile']).any()

if overlap:
    print("❌ DataFrames are NOT mutually exclusive.")
else:
    print("✅ DataFrames are mutually exclusive.")

✅ DataFrames are mutually exclusive.


In [35]:
# Convert to sets and check for intersection
set1 = set(df_export_control['customer_mobile'])
set2 = set(df_export['customer_mobile'])

if set1.isdisjoint(set2):
    print("✅ DataFrames are mutually exclusive.")
else:
    print("❌ DataFrames are NOT mutually exclusive.")

✅ DataFrames are mutually exclusive.


In [36]:
df_export_control[df_export_control['customer_mobile'].isin(['7639345071', '9785494112', '8108282883', '7983804744', '9845095045'])]

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,mobile_length,group_tc
199558,6288c6694281e53bbe33d133,Bangalore,7639345071,25,7,0,11,5,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,10,control
205088,62c7404566b8a31b4329f965,Bangalore,7983804744,76,17,2,26,29,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,10,control


In [37]:
df_export_control = df_export_control[~df_export_control['customer_mobile'].isin(['7639345071', '9785494112', '8108282883', '7983804744', '9845095045'])]

In [38]:
df_export_control[df_export_control['customer_mobile'].isin(['7639345071', '9785494112', '8108282883', '7983804744', '9845095045'])]

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,mobile_length,group_tc


In [39]:
df_export[df_export['customer_mobile'].isin(['7639345071', '9785494112', '8108282883', '7983804744', '9845095045'])]

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,mobile_length,group_tc
617402,60dd7ab509aa99d4c33d1dd5,Bangalore,9785494112,202,118,6,31,29,PHH,ELITE,COMMITTED,ONLY_AUTO,MALE,10,test


In [48]:
# The new row to add
new_row = pd.DataFrame({'customer_mobile': ['7639345071', '9785494112', '8108282883', '7983804744', '9845095045'],
                        'customer_id' : ['6288c6694281e53bbe33d133', '60dd7ab509aa99d4c33d1dd5', '5a2774b1178fbb6f92482223', '62c7404566b8a31b4329f965', '59b3a2bd840dfa9e40342308'],
                        'group_tc' : ['test', 'test', 'test', 'test', 'test']
                       })

# Append it
df_export_test = pd.concat([df_export, new_row], ignore_index=True)

df_export_test

,customer_id,city_name,customer_mobile,gross_count,net_count,cobrm,cobra,ocara,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,mobile_length,group_tc
0,663b4f0aad80a754bf3a81eb,Bangalore,9025479400,1.0,1.0,0.0,0.0,0.0,PHH,DORMANT,DORMANT,UNKNOWN,FEMALE,10.0,test
1,6795f673c46f6c4f7eef9e9c,Bangalore,7892031156,5.0,5.0,0.0,0.0,0.0,PHH,DORMANT,DORMANT,UNKNOWN,MALE,10.0,test
2,65bdf23de65856c258cc311b,Bangalore,7797556557,74.0,67.0,0.0,4.0,3.0,PHH,ELITE,COMMITTED,ONLY_AUTO,MALE,10.0,test
3,5e494571ab02656b352834d7,Bangalore,9051855749,99.0,65.0,0.0,23.0,9.0,PHH,ELITE,COMMITTED,NO_AFFINITY,MALE,10.0,test
4,5d5bc0b2bde10e0d54496ffa,Bangalore,9390267572,8.0,4.0,0.0,3.0,1.0,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,10.0,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449999,6288c6694281e53bbe33d133,NaN,7639345071,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,test
450000,60dd7ab509aa99d4c33d1dd5,NaN,9785494112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,test
450001,5a2774b1178fbb6f92482223,NaN,8108282883,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,test
450002,62c7404566b8a31b4329f965,NaN,7983804744,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,test


In [51]:
display(df_export_test[['customer_id', 'customer_mobile', 'group_tc']])
display(df_export_control[['customer_id', 'customer_mobile', 'group_tc']])

,customer_id,customer_mobile,group_tc
0,663b4f0aad80a754bf3a81eb,9025479400,test
1,6795f673c46f6c4f7eef9e9c,7892031156,test
2,65bdf23de65856c258cc311b,7797556557,test
3,5e494571ab02656b352834d7,9051855749,test
4,5d5bc0b2bde10e0d54496ffa,9390267572,test
...,...,...,...
449999,6288c6694281e53bbe33d133,7639345071,test
450000,60dd7ab509aa99d4c33d1dd5,9785494112,test
450001,5a2774b1178fbb6f92482223,8108282883,test
450002,62c7404566b8a31b4329f965,7983804744,test


,customer_id,customer_mobile,group_tc
0,67a62a5739dd551af0eddb23,6202742044,control
1,61d00a0da757467724e4a143,7975903747,control
2,63e4ea9fec0ea850e32cdbd1,7845811202,control
3,65ae2b4b35ebfff90f5fc774,9019388700,control
4,6768ee424e77d8694bd7b467,7047699844,control
...,...,...,...
449995,670e291efc39bb3024f3e9d9,9994397130,control
449996,65474f8a7a091bf986def29b,8867345603,control
449997,5cda9143d2ee6e6cf2e8a0e8,9914646655,control
449998,647c270505281a44c9ff0487,7848999824,control


In [56]:
df_export_test[['customer_id', 'customer_mobile', 'group_tc']].drop_duplicates().to_csv('loyalty_experiment1_test_group.csv', header=False, index=False)
df_export_control[['customer_id', 'customer_mobile', 'group_tc']].to_csv('loyalty_experiment1_controt_group.csv', header=False, index=False)

In [54]:
df_export_test[['customer_id', 'customer_mobile', 'group_tc']].drop_duplicates().shape

(450003, 3)

In [55]:
df_export_test[['customer_id', 'customer_mobile', 'group_tc']].shape

(450004, 3)

In [49]:
df_export[['customer_mobile']].to_csv(local_datasource + '/test_group_v1.csv', header=False, index=False)
df_export_control[['customer_mobile']].to_csv(local_datasource + '/control_group_v1.csv', header=False, index=False)

In [50]:
# total rows in df
total_rows = len(df_export_test)

# split sizes
split_size = 150000

# split the dataframe
df_export_test_df1 = df_export_test.iloc[:split_size]
df_export_test_df2 = df_export_test.iloc[split_size:split_size*2]
df_export_test_df3 = df_export_test.iloc[split_size*2:]

In [51]:
print(df_export_test_df1.shape)
print(df_export_test_df2.shape)
print(df_export_test_df3.shape)

(150000, 15)
(150000, 15)
(150004, 15)


In [52]:
# total rows in df
total_rows = len(df_export_control)

# split sizes
split_size = 150000

# split the dataframe
df_export_control_df1 = df_export_control.iloc[:split_size]
df_export_control_df2 = df_export_control.iloc[split_size:split_size*2]
df_export_control_df3 = df_export_control.iloc[split_size*2:]

In [53]:
print(df_export_control_df1.shape)
print(df_export_control_df2.shape)
print(df_export_control_df3.shape)

(150000, 15)
(150000, 15)
(149997, 15)


In [54]:
df_export_control_df1[['customer_mobile']].to_csv(local_datasource + '/final/control_group_t1.csv', index=False)
df_export_control_df2[['customer_mobile']].to_csv(local_datasource + '/final/control_group_t2.csv', index=False)
df_export_control_df3[['customer_mobile']].to_csv(local_datasource + '/final/control_group_t3.csv', index=False)
df_export_test_df1[['customer_mobile']].to_csv(local_datasource + '/final/test_group_t1.csv', index=False)
df_export_test_df2[['customer_mobile']].to_csv(local_datasource + '/final/test_group_t2.csv', index=False)
df_export_test_df3[['customer_mobile']].to_csv(local_datasource + '/final/test_group_t3.csv', index=False)

In [55]:
df_export_control_df1 = pd.read_csv(local_datasource + '/final/control_group_t1.csv')
df_export_control_df2 = pd.read_csv(local_datasource + '/final/control_group_t2.csv')
df_export_control_df3 = pd.read_csv(local_datasource + '/final/control_group_t3.csv')
df_export_test_df1 = pd.read_csv(local_datasource + '/final/test_group_t1.csv')
df_export_test_df2 = pd.read_csv(local_datasource + '/final/test_group_t2.csv')
df_export_test_df3 = pd.read_csv(local_datasource + '/final/test_group_t3.csv')

In [56]:
display(df_export_control_df1.head(1))
display(df_export_control_df2.head(1))
display(df_export_control_df3.head(1))
display(df_export_test_df1.head(1))
display(df_export_test_df2.head(1))
display(df_export_test_df3.head(1))

,customer_mobile
0,6202742044


,customer_mobile
0,9880439845


,customer_mobile
0,6366130511


,customer_mobile
0,9025479400


,customer_mobile
0,9487525882


,customer_mobile
0,6302444573


### Final

In [57]:
df_export_control_df1.to_csv(local_datasource + '/final/headerless/control_group_t1.csv', header=False, index=False)
df_export_control_df2.to_csv(local_datasource + '/final/headerless/control_group_t2.csv', header=False, index=False)
df_export_control_df3.to_csv(local_datasource + '/final/headerless/control_group_t3.csv', header=False, index=False)
df_export_test_df1.to_csv(local_datasource + '/final/headerless/test_group_t1.csv', header=False, index=False)
df_export_test_df2.to_csv(local_datasource + '/final/headerless/test_group_t2.csv', header=False, index=False)
df_export_test_df3.to_csv(local_datasource + '/final/headerless/test_group_t3.csv', header=False, index=False)

In [58]:
test_df = pd.read_csv(local_datasource + '/test_group_v1.csv')
control_df = pd.read_csv(local_datasource + '/control_group_v1.csv')

In [59]:
test_df

,9025479400
0,7892031156
1,7797556557
2,9051855749
3,9390267572
4,9495442064
...,...
449993,7904025232
449994,9916362966
449995,9003261320
449996,9731142819


In [60]:
control_df

,6202742044
0,7975903747
1,7845811202
2,9019388700
3,7047699844
4,9902388180
...,...
449991,9994397130
449992,8867345603
449993,9914646655
449994,7848999824


In [61]:
overlap1 = control_df['6202742044'].isin(test_df['9025479400']).any()

if overlap1:
    print("❌ DataFrames are NOT mutually exclusive.")
else:
    print("✅ DataFrames are mutually exclusive.")

✅ DataFrames are mutually exclusive.
